## Finding all adverbials from a database

The aim of this notebook is to find all adverbials from the Estonian Reference corpus. This is needed to annotate them with semantic class using both rule based methods and LLMs. The code extracts data from Katrin Tsepelina's database [v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db](https://github.com/estnltk/syntax_experiments/tree/verb_templates/workflows/001_verb_transactions/v33) with data extracted from the Estonian Reference corpus.

This code creates a table for GPT labelled data based on the tag.

In [1]:
#imports
import sqlite3
import pandas as pd
from tqdm import tqdm

In [73]:
# database file path
DB_FILE = "../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310_gpt_labelled.db"

TARGET_TAG = "S"

CLASS = "n50"

GPT_TABLE = f"gpt_labelled_{TARGET_TAG}_{CLASS}"

# column for final new tag
TAG_COL = "gpt_tags"

In [3]:
def column_exists(cursor, table, column):
    cursor.execute(f"PRAGMA table_info({table})")
    return any(row[1] == column for row in cursor.fetchall())

In [49]:
# connecting with database
#conn = sqlite3.connect(DB_FILE)
#cursor = conn.cursor()

### Create new table

In [74]:
file = f"../data/datasets_500_all_results/finished/{TARGET_TAG}_{CLASS}_500_batch_tagged_final.csv"

In [75]:
df = pd.read_csv(file, sep=",", encoding="utf-8")
df["initial_cat"] = CLASS
df["id"] = df.index
df["verb_compound"] = df["verb_compound"].fillna("")

In [76]:
len(df)

5254

In [26]:
#df.head(3)

In [77]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

df.to_sql(GPT_TABLE, conn, if_exists="replace", index=False)

conn.close()

### Create tags column based on gpt predicted tag

For tag combinations:

- EVENT - E
- TIME - T
- LOC - L
- LOCEVENT - EL
- LOCEVENTTIME - ELT
- ALIVE - A
- STATE - S

In [78]:

combinations = { 
    "T": ["T","LT","ET","ELT","AT","ALT","AET","LST","AST","ST"], 
    "L": ["L","LT","EL","ELT","LS","AL","ALT","LST"], 
    "E": ["E","EL","ET","ELT","AE","AET",], 
    "A": ["A","AT","AL","AE","AS","ALT","AET","AST"], 
    "S": ["S","LS","AS","LST","AST","ST",], 
}


def get_combos(letters, limit):
    letters_set = set(letters)
    result = []
    for letter in letters:
        for combo in combinations[letter]:
            # combo has the right amount of tags and not other tags
            if len(combo) > limit and letters_set.issubset(combo):
                result.append(combo)

    return result


def get_tags(gpt_tag):

    sources = []
    # Info about what tags are present
    if gpt_tag is not None:
        sources.append(gpt_tag)

    if not sources:
        return []

    letters = set(sources)

    if len(letters) == 1: # only one tag is present, get 1,2,3 letter combinations
        tags = combinations[next(iter(letters))]
    else: # more than one tag, get 2+3 or just 3 letter combinations
        limit = 1 if len(letters) == 2 else 2
        tags = list(set(get_combos(letters, limit)))

    # alphabetically and order by 1, letter tags, 2-letter tags, 3-letter tags
    tags.sort(key=lambda w: (len(w), w))
    return tags

In [79]:
# connecting with database
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

In [80]:
# get necessary columns from obl table
query = f"SELECT id, tag FROM {GPT_TABLE}"
obl_tags = pd.read_sql(query, conn)
#obl_tags # 375,810 rows

In [81]:
# Get tags for all rows

tags_col = []

for i in tqdm(range(len(obl_tags))):
    idx = obl_tags.iloc[i]["id"]
    gpt = obl_tags.iloc[i]["tag"]

    tag_list = get_tags(gpt)
    if len(tag_list) != 0:
        tag_str = "|" + "|".join(tag_list) + "|"
    else:
        tag_str = ""
        
    tags_col.append((idx, tag_str))

tags_col_clean = [(int(i), t) for i, t in tags_col]
assert len(tags_col_clean) == len(obl_tags)

100%|████████████████████████████████████| 5254/5254 [00:00<00:00, 16198.63it/s]


In [82]:

# New column if it doesn't exist
if not column_exists(cursor, GPT_TABLE, TAG_COL):
    cursor.execute("ALTER TABLE " + GPT_TABLE +  f" ADD COLUMN {TAG_COL} TEXT")

# Create a temporary table
cursor.execute("CREATE TEMP TABLE temp_tags (id INT PRIMARY KEY, tags TEXT)")

# Insert all values into the temp table
cursor.executemany("INSERT INTO temp_tags (id, tags) VALUES (?, ?)", tags_col_clean)

# Perform a fast join-based update
query = f"""
    UPDATE {GPT_TABLE}
    SET {TAG_COL} = (
        SELECT tags
        FROM temp_tags
        WHERE temp_tags.id = {GPT_TABLE}.id
        LIMIT 1
     )
    WHERE EXISTS (
        SELECT 1
        FROM temp_tags
        WHERE temp_tags.id = {GPT_TABLE}.id
    )
"""

cursor.execute(query)

conn.commit()
conn.close()

In [83]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

query = f"SELECT * FROM {GPT_TABLE} limit 10"
res = pd.read_sql(query, conn)

conn.close()

In [84]:
res

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,...,E,T,L1,L2,L,S,tag,initial_cat,id,gpt_tags
0,10615189,17014059,1,sündima,,el,kontekst,Kontekstist,"Kontekstist lahtikistuna , iseseisva orkestrit...",None,...,no,no,no,yes,yes,no,L,n50,0,|L|AL|EL|LS|LT|ALT|ELT|LST|
1,9201168,14788766,9,veenduma,,in,tagasivalimine,tagasivalimises,Saksamaa praegune liidukantsler Helmut Kohl on...,None,...,yes,no,no,no,no,no,E,n50,1,|E|AE|EL|ET|AET|ELT|
2,10602658,16995259,6,tulenema,,el,favoriit,Favoriidist,""" See ei tulene mitte Favoriidist , vaid minus...",None,...,no,no,yes,no,yes,no,L,n50,2,|L|AL|EL|LS|LT|ALT|ELT|LST|
3,7624824,12231702,3,sündima,,el,enesedistsipliin,enesedistsipliinist,"Enesekindlus sünnib enesedistsipliinist , õpet...",None,...,no,no,yes,no,yes,no,L,n50,3,|L|AL|EL|LS|LT|ALT|ELT|LST|
4,3765694,6065540,7,tulenema,,el,ehitus,ehitusest,Ehitustegevuse kasv tulenes elamu- ja infrastr...,None,...,yes,no,no,no,no,no,E,n50,4,|E|AE|EL|ET|AET|ELT|
5,17646522,26978576,16,viitama,,all,antisotsiaalsus,antisotsiaalsusele,"noh , seal peab olema põhjus , miks süütu olla...",None,...,no,no,no,no,no,yes,S,n50,5,|S|AS|LS|ST|AST|LST|
6,17970721,27344992,8,kahtlema,,in,pauk,Paugus,Tõsiseltvõetavatest astronoomidest ei kahtle t...,None,...,yes,no,no,no,no,no,E,n50,6,|E|AE|EL|ET|AET|ELT|
7,7661626,12284598,3,sattuma,,adit,ajalehetoimetus,ajalehetoimetusse,"Kuidas materjal ajalehetoimetusse sattus , pol...",None,...,no,no,yes,no,yes,no,L,n50,7,|L|AL|EL|LS|LT|ALT|ELT|LST|
8,7212766,11601530,15,viitama,,all,arengutee,arenguteele,"Tal pole veel kindlust , millega läbida ükstei...",None,...,no,no,yes,no,yes,no,L,n50,8,|L|AL|EL|LS|LT|ALT|ELT|LST|
9,18016939,27415059,6,tulenema,,el,analüüsimine,analüüsimisest,Need numbrid tulenevad BRS-i äriplaani analüüs...,None,...,yes,no,no,no,no,no,E,n50,9,|E|AE|EL|ET|AET|ELT|


## Add transaction_head table if missing

In [46]:
DB2 = "../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310_test.db"
               
conn = sqlite3.connect(DB2)

query = f"SELECT * FROM transaction_head"
res = pd.read_sql(query, conn)

conn.close()



conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

res.to_sql("transaction_head", conn, if_exists="replace", index=False)

conn.commit()
conn.close()



DB2 = "../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310_test.db"
DB3 = "../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310_orig.db"   

#conn = sqlite3.connect(DB2)
#query = f"SELECT * FROM transaction_head"
#res = pd.read_sql(query, conn)
#conn.close()

conn = sqlite3.connect(DB3)
cursor = conn.cursor()

res.to_sql("transaction_head", conn, if_exists="replace", index=False)

conn.commit()
conn.close()